<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Evidence-Verified-RAG-Workflow-/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Ollama

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,147 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama Server in background

In [4]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for the server to start
time.sleep(5)
print("Ollama server started, PID:", process.pid)


Ollama server started, PID: 6438


#Pull LIama 3.1

In [5]:
!ollama pull llama3.1:8b

#Pull the embedding model

In [6]:
!ollama pull nomic-embed-text

# Verify both are installed

In [7]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.1:8b                46e0c10c039e    4.9 GB    5 seconds ago             


#Test generation works

In [8]:
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say 'setup successful' and nothing else.",
        "stream": False,
    }
)
print(response.json()["response"])

Setup successful


#Install Langchain+dependencies for next steps

In [9]:
!pip install -q langchain langchain-community langchain-ollama faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Connect Langchain to Ollama

In [10]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1:8b", temperature=0.2)
result = llm.invoke("In one sentence, what is Retrieval Augmented Generation?")
print(result)

Retrieval Augmented Generation (RAG) is a type of artificial intelligence model that combines the strengths of retrieval-based and generation-based approaches, where a model first retrieves relevant information from a large database and then generates new text based on that information.


#PDF Ingestion + Chunking

## Upload a test PDF to colab

In [11]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving dl based detecting malicious packages in pypl.pdf to dl based detecting malicious packages in pypl.pdf
Saving On the Performance of Malware Detection.pdf to On the Performance of Malware Detection.pdf
Saving Quantum redy crypto.pdf to Quantum redy crypto.pdf
Uploaded: dl based detecting malicious packages in pypl.pdf


#Install the text-splitter package

In [14]:
!pip install -q langchain langchain-community langchain-ollama  langchain-text-splitters faiss-cpu pypdf

##Load the PDF with PyPDFLoader

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print("Sample text from page 1:\n", pages[0].page_content[:500])

Loaded 26 pages
Sample text from page 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The s


##Chunk the text with RecursiveChracterTextSplitter

In [20]:
!pip install -q -U langchain-text-splitters


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("import successful!")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ",", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunks")
print("Sample text from chunk 1:\n", chunks[0].page_content[:300])

import successful!
Split into 147 chunks
Sample text from chunk 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan Uni


##Quick sanity check -- Inspect a few chunks

In [23]:
for i in range(min(3, len(chunks))):
  print(f"---chunk {i} ---")
  print(chunks[i].page_content[:200])
  print("Metadata:", chunks[i].metadata)
  print()

---chunk 0 ---
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
R
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-04-30T00:20:09+00:00', 'author': 'Sk Tanzir Mehedi; Raja Jurdak; Chadni Islam; Abu Bakar Siddique Mahi; Gowri Ramachandran', 'doi': 'https://doi.org/10.48550/arXiv.2604.26219', 'keywords': 'Deep learning, malicious detection, software supply chain security, dynamic analysis, stability, explainable AI.', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2026-04-30T00:20:09+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem', 'trapped': '/False', 'arxivid': 